In [100]:
##### Assembles a shapefile with all sub-national geographies for capital stock data

import os
import pandas as pd
import geopandas as gpd

In [101]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
capital_stock = pd.read_csv(f"{cd}/Data/Clean/Capital_stock/subnational_capital_stock_final.csv")

# Set save path
save_path = f"{cd}/Data/Clean/Capital_stock/subnational_capital_stock_final_GEO.shp"

In [102]:
### Prep geographies

capital_stock['GEO_ID'] = capital_stock['GEO_ID'].astype('str')

# EU
EU_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/EU_FADN/EU_FADN_2021_regions.shp")
EU_geo['FADN_code_'] = EU_geo['FADN_code_'].astype('str')
EU_geo = EU_geo.merge(capital_stock, left_on='FADN_code_', right_on='GEO_ID', how='inner')

# US
US_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/US_counties/cb_2018_us_county_20m.shp")
US_geo['GEOID'] = US_geo['GEOID'].astype('str')
capital_stock_US = capital_stock.loc[capital_stock['ISO3'] == 'USA'].copy()
capital_stock_US["GEO_ID"] = capital_stock_US["GEO_ID"].astype(str).str.zfill(5)
US_geo = US_geo.merge(capital_stock_US, left_on='GEOID', right_on='GEO_ID', how='inner')

# Canada
CAN_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/Canadian_CCS/lccs000a21a_e.shp")
CAN_geo['CCSUID'] = CAN_geo['CCSUID'].astype('str')
CAN_geo = CAN_geo.merge(capital_stock, left_on='CCSUID', right_on='GEO_ID', how='inner')

# China
CHN_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/China_provinces/chn_admbnda_adm1_ocha_2020.shp")
CHN_geo['ADM1_PCODE'] = CHN_geo['ADM1_PCODE'].astype('str')
CHN_geo = CHN_geo.merge(capital_stock, left_on='ADM1_PCODE', right_on='GEO_ID', how='inner')

# Mexico
MEX_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/MEX_state/mex_admin1.shp")
MEX_geo['adm1_pcode'] = MEX_geo['adm1_pcode'].astype('str')
MEX_geo = MEX_geo.merge(capital_stock, left_on='adm1_pcode', right_on='GEO_ID', how='inner')

# Brazil
BRA_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/Brazil_Municipalities/BR_Municipios_2023.shp")
BRA_geo['CD_MUN'] = BRA_geo['CD_MUN'].astype('str')
BRA_geo = BRA_geo.merge(capital_stock, left_on='CD_MUN', right_on='GEO_ID', how='inner')

# Egypt
EGY_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/Egypt_Governorates/EGY_governorates.shp")
EGY_geo['HASC_1'] = EGY_geo['HASC_1'].astype('str')
EGY_geo = EGY_geo.merge(capital_stock, left_on='HASC_1', right_on='GEO_ID', how='inner')

# Ghana
GHA_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/Ghana_regions/GHA_regions.shp")
GHA_geo['ISO_1'] = GHA_geo['ISO_1'].astype('str')
GHA_geo = GHA_geo.merge(capital_stock, left_on='ISO_1', right_on='GEO_ID', how='inner')

# India
IND_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/India/India_states.shp")
# IND_geo['GID_1'] = IND_geo['GID_1'].astype('str')
# IND_geo = IND_geo.merge(capital_stock, left_on='GID_1', right_on='GEO_ID', how='inner')

# Turkey
TUR_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/Turkey/TUR_provinces.shp")
TUR_geo['HASC_1'] = TUR_geo['HASC_1'].astype('str')
TUR_geo = TUR_geo.merge(capital_stock, left_on='HASC_1', right_on='GEO_ID', how='inner')

# South Africa
ZAF_geo = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/South_Africa_municipalities/ZAF_municipalities.shp")
ZAF_geo['CC_3'] = ZAF_geo['CC_3'].astype('str')
ZAF_geo = ZAF_geo.merge(capital_stock, left_on='CC_3', right_on='GEO_ID', how='inner')


In [103]:
### Filter geographies to needed data and align CRS

col_to_keep = ['ISO3', 'GEO_ID', 'GEO_ID_NAME', 'ag_capital_stock_USD_nominal', 'geometry']

geo_gdfs = [CAN_geo, CHN_geo, EU_geo, MEX_geo, US_geo, BRA_geo, EGY_geo, GHA_geo, TUR_geo, ZAF_geo]

for df in geo_gdfs:
    df.drop(columns=[c for c in df.columns if c not in col_to_keep], inplace=True)

# Align CRS
geo_gdfs = [df.to_crs(4326) for df in geo_gdfs]


In [104]:
### Combine all geodf's and save

full_geo = gpd.GeoDataFrame(
    pd.concat(geo_gdfs, ignore_index=True),
    crs=geo_gdfs[0].crs
)

# final clean to save as SHP
full_geo['ag_capital_stock_USD_nominal'] = full_geo['ag_capital_stock_USD_nominal'].round(0)
full_geo['value'] = full_geo['ag_capital_stock_USD_nominal'] / 1e6
full_geo = full_geo.drop(columns=['ag_capital_stock_USD_nominal'])
full_geo['unit'] = 'ag_capital_stock_million_USD_nominal'
full_geo = full_geo.rename(columns={'GEO_ID_NAME': 'GEO_ID_NM'})

full_geo.to_file(save_path)